# U03 SQL（二）：join、聚合、子查詢與 window functions

**資料庫管理**・統計系三年級・09/24　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

把表接起來、把資料摺起來、再攤開來排名——**課末公布專題指派表** ★

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 | 與專題的關係 |
|---|---|---|---|
| 第 1 節 | 50 | join 全家（inner／left／full／self）、ON vs WHERE 陷阱、**重複計數陷阱**、集合運算 | 你的應用每個畫面都在 join |
| 第 2 節前半 | 25 | `GROUP BY`／`HAVING`、條件式聚合（樞紐）、子查詢四型、CTE、**遞迴 CTE** | 報表的骨架 |
| 第 2 節後半 | 25 | **window functions**：排名、差分、移動平均、分位、累積、佔比 | 專題要求「≥1 個 window 報表」 |
| 實作 | 25 | 電商資料分析 8 題＋挑戰加碼 | 專題報表的直接演練 |
| ★ 課末 | 10 | **公布專題指派表**＋題目導覽 | 回家精讀自己的題目 |

兩個資料庫：熟悉的 **univ.db**（小，看得清楚）＋新朋友 **sales.db**（10 萬列電商訂單，才有分析的感覺）。

In [ ]:
#@title 📦 資料準備：建立課程範例資料庫 univ.db（先跑它，別急著讀懂——本單元結束你就全看得懂）
import sqlite3, os, pandas as pd

if os.path.exists("univ.db"):
    os.remove("univ.db")
con = sqlite3.connect("univ.db")
con.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE student(
  sid   TEXT PRIMARY KEY,          -- 學號
  name  TEXT NOT NULL,             -- 姓名
  dept  TEXT NOT NULL,             -- 系所
  year  INTEGER CHECK(year BETWEEN 1 AND 4)   -- 年級
);
CREATE TABLE instructor(
  iid   TEXT PRIMARY KEY,
  name  TEXT NOT NULL,
  dept  TEXT NOT NULL,
  salary REAL
);
CREATE TABLE course(
  cid     TEXT PRIMARY KEY,
  title   TEXT NOT NULL,
  dept    TEXT NOT NULL,
  credits INTEGER NOT NULL DEFAULT 3
);
CREATE TABLE takes(                -- 修課紀錄
  sid TEXT REFERENCES student(sid),
  cid TEXT REFERENCES course(cid),
  semester TEXT,                   -- 學期，如 114-1
  grade REAL,                      -- NULL = 在修中
  PRIMARY KEY (sid, cid, semester)
);
CREATE TABLE teaches(              -- 授課紀錄
  iid TEXT REFERENCES instructor(iid),
  cid TEXT REFERENCES course(cid),
  semester TEXT,
  PRIMARY KEY (iid, cid, semester)
);
""")
con.executemany("INSERT INTO student VALUES (?,?,?,?)", [
 ("S001","林佳蓉","統計",3),("S002","陳威廷","統計",3),("S003","張雅筑","統計",3),
 ("S004","李承翰","統計",2),("S005","王思穎","統計",4),("S006","黃冠宇","統計",3),
 ("S007","吳孟軒","資訊",3),("S008","劉子涵","資訊",2),("S009","蔡明修","資訊",4),
 ("S010","許芷瑄","數學",3),("S011","鄭宇翔","數學",2),("S012","謝欣妤","數學",4),
 ("S013","洪偉倫","企管",3),("S014","郭品妍","企管",2),("S015","曾柏勳","企管",3),
 ("S016","賴韻如","統計",1),("S017","周家豪","資訊",1),("S018","江美慧","統計",4),
 ("S019","趙國彬","數學",3),("S020","方語彤","企管",4),
])
con.executemany("INSERT INTO instructor VALUES (?,?,?,?)", [
 ("I01","王教授","統計",118000.0),("I02","李教授","統計",102000.0),
 ("I03","張教授","資訊",111000.0),("I04","陳教授","數學",96000.0),
 ("I05","林教授","企管",99000.0),("I06","徐教授","統計",None),
])
con.executemany("INSERT INTO course VALUES (?,?,?,?)", [
 ("C101","統計學（一）","統計",3),("C102","迴歸分析","統計",3),
 ("C103","資料庫管理","統計",3),("C104","機率論","統計",3),
 ("C201","微積分","數學",4),("C202","線性代數","數學",3),
 ("C301","程式設計","資訊",3),("C302","資料結構","資訊",3),
])
con.executemany("INSERT INTO takes VALUES (?,?,?,?)", [
 ("S001","C101","114-1",88),("S001","C104","114-1",92),("S001","C102","114-2",85),
 ("S001","C103","115-1",None),("S002","C101","114-1",76),("S002","C102","114-2",81),
 ("S002","C103","115-1",None),("S003","C101","114-1",95),("S003","C104","114-1",89),
 ("S003","C102","114-2",91),("S003","C103","115-1",None),("S004","C101","114-2",67),
 ("S004","C201","114-2",72),("S004","C104","115-1",None),("S005","C101","113-1",82),
 ("S005","C102","113-2",78),("S005","C103","114-1",90),("S006","C101","114-1",58),
 ("S006","C104","114-1",61),("S006","C103","115-1",None),("S007","C301","114-1",93),
 ("S007","C302","114-2",87),("S007","C103","115-1",None),("S008","C301","114-2",74),
 ("S008","C302","115-1",None),("S009","C301","113-1",85),("S009","C302","113-2",80),
 ("S009","C202","114-1",77),("S010","C201","114-1",90),("S010","C202","114-2",94),
 ("S011","C201","114-2",63),("S011","C202","115-1",None),("S012","C201","113-1",71),
 ("S012","C202","113-2",75),("S012","C101","114-1",79),("S013","C101","114-1",70),
 ("S013","C103","115-1",None),("S014","C101","114-2",55),("S015","C101","114-1",83),
 ("S015","C102","114-2",88),("S016","C101","115-1",None),("S017","C301","115-1",None),
 ("S018","C101","113-1",96),("S018","C102","113-2",93),("S018","C104","114-1",98),
 ("S018","C103","114-1",94),("S019","C201","114-1",84),("S019","C202","114-2",86),
 ("S020","C101","113-2",73),("S020","C102","114-1",69),
])
con.executemany("INSERT INTO teaches VALUES (?,?,?)", [
 ("I01","C101","114-1"),("I01","C101","114-2"),("I01","C104","114-1"),
 ("I02","C102","114-2"),("I02","C102","114-1"),("I06","C103","115-1"),
 ("I06","C103","114-1"),("I03","C301","114-1"),("I03","C302","114-2"),
 ("I04","C201","114-1"),("I04","C202","114-2"),("I05","C101","113-2"),
])
con.commit()

def q(sql, params=()):
    """跑一句 SELECT，回傳 pandas DataFrame（Colab 會漂亮顯示）"""
    return pd.read_sql_query(sql, con, params=params)

for t in ["student","instructor","course","takes","teaches"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<12}{n:>4} 列")
print("univ.db 就緒 ✅")

# 第 1 節：join——把正規化拆開的表接回來

## 1.1 為什麼需要 join？

U04 會教你把資料**拆表存**（避免重複與異常）；代價是查詢時要**接回來**。join 就是那個「接」。

概念起點是 **cross join**（笛卡兒積）：左表每列 × 右表每列，全部配對。單獨用幾乎沒意義，但它是所有 join 的數學基底——**inner join ＝ cross join ＋ 配對條件過濾**。

In [ ]:
print("student 20 列 × course 8 列 = cross join",
      q("SELECT COUNT(*) c FROM student CROSS JOIN course").iloc[0,0], "列（全是無意義的配對）")
q("SELECT s.name, c.title FROM student s CROSS JOIN course c LIMIT 5")   # 看一眼長相

In [ ]:
# cross join 的正經用途：先生成「全組合」、再 LEFT JOIN 找「缺了誰」——修課矩陣的補洞
# 例：統計系每位學生 × C101，誰「還沒修過」C101？
q("""SELECT s.sid, s.name,
        CASE WHEN t.sid IS NULL THEN '❌ 未修' ELSE '✔ 修過/在修' END AS c101
     FROM student s
     LEFT JOIN takes t ON s.sid = t.sid AND t.cid = 'C101'
     WHERE s.dept = '統計'
     ORDER BY c101, s.sid""")
# 期望：0 位未修。點名表、應到未到、庫存盤點缺漏——全是「全組合 − 實際」這一招

## 1.2 INNER JOIN：只留「配對成功」的列

```sql
SELECT s.name, c.title, t.grade
FROM takes t
JOIN student s ON t.sid = s.sid        -- 「t 的 sid 要對上 s 的 sid」
JOIN course  c ON t.cid = c.cid        -- 可以一路接下去
```

- `t`、`s`、`c` 是**表別名**——多表查詢的基本禮貌，欄位一律寫 `別名.欄位`。
- `ON` 放**配對條件**；一般過濾條件放 `WHERE`（等會看混用的陷阱）。
- 老派寫法 `FROM a, b WHERE a.x = b.x` 效果相同，但忘了 WHERE 就變 cross join——**一律用顯式 JOIN**。
- （也有 `USING(sid)`、`NATURAL JOIN` 的縮寫式——課堂不用：NATURAL 會拿「所有同名欄」亂配，schema 一改就出鬼故事，AI 生出來請改寫成 ON。）

**多表 join 的心法：一次接一張。** 先 `takes ⋈ student` 看結果對不對，再接 `course`——像組樂高，別一口氣蓋完才發現歪了。

In [ ]:
# 成績單：三表 join——這就是「應用畫面」的原型（每個 App 頁面背後都是一句 join）
q("""SELECT s.sid, s.name AS 學生, c.title AS 課程, t.semester AS 學期, t.grade AS 成績
     FROM takes t
     JOIN student s ON t.sid = s.sid
     JOIN course  c ON t.cid = c.cid
     ORDER BY s.sid, t.semester
     LIMIT 10""")

In [ ]:
# join ＋ 一般過濾：統計系學生「有成績」的修課，90 分以上
q("""SELECT s.name, c.title, t.grade
     FROM takes t
     JOIN student s ON t.sid = s.sid
     JOIN course  c ON t.cid = c.cid
     WHERE s.dept = '統計' AND t.grade >= 90
     ORDER BY t.grade DESC""")

## 1.3 LEFT JOIN：左表一個都不能少

```
INNER：只留配對成功        LEFT：左表全留，右邊沒對到 → 補 NULL
  A ∩ B                      A 全部（B 缺的位置是 NULL）
```

兩大用途：
1. **報表不能漏人**：「每位學生的修課數」——沒修課的也要出現（0 筆）。
2. **找『沒有』的東西（anti-join）**：右邊是 NULL 的列＝沒配到＝「從未⋯⋯」。

In [ ]:
# 用途 1 實演：每位學生的 115-1 修課數——INNER 會讓 0 筆的人消失，LEFT 保住全員
q("""SELECT s.sid, s.name, COUNT(t.cid) AS n_115_1     -- COUNT(欄) 不數 NULL → 沒修的自然是 0
     FROM student s
     LEFT JOIN takes t ON s.sid = t.sid AND t.semester = '115-1'
     GROUP BY s.sid
     ORDER BY n_115_1 DESC, s.sid LIMIT 8""")
# 報表鐵則：「每個 X 的…」開頭的需求，X 那張表當左表、LEFT JOIN 伺候

In [ ]:
# 用途 2：anti-join——這學期（115-1）「沒有」修任何課的學生
q("""SELECT s.sid, s.name, s.dept
     FROM student s
     LEFT JOIN takes t ON s.sid = t.sid AND t.semester = '115-1'
     WHERE t.sid IS NULL
     ORDER BY s.sid""")
# 期望：9 位

## 1.4 本課最重要的陷阱：LEFT JOIN 的條件放 ON 還是 WHERE？

「每位學生**這學期**的修課」——兩種寫法，結果天差地遠：

| 寫法 | 效果 | 列數 |
|---|---|---|
| `LEFT JOIN takes t ON s.sid=t.sid AND t.semester='115-1'` | ✅ 先用條件配對，**左表 20 人全留**，沒修的補 NULL | 20 |
| `LEFT JOIN takes t ON s.sid=t.sid WHERE t.semester='115-1'` | ❌ 先全配對，**WHERE 再殺掉所有 NULL 列** → 悄悄變 INNER JOIN | 11 |

心法：**對右表的過濾條件，放 ON**（參與配對）；**對左表或最終結果的過濾，放 WHERE**。
AI 生成的 SQL 特別常犯這個錯——現在你有能力抓它了。

In [ ]:
on_ver    = q("""SELECT s.sid, t.cid FROM student s
                  LEFT JOIN takes t ON s.sid = t.sid AND t.semester='115-1'""")
where_ver = q("""SELECT s.sid, t.cid FROM student s
                  LEFT JOIN takes t ON s.sid = t.sid WHERE t.semester='115-1'""")
print(f"條件在 ON：{len(on_ver)} 列（20 人全在，沒修的 cid 是 NaN/None）")
print(f"條件在 WHERE：{len(where_ver)} 列（沒修課的 9 人被 WHERE 滅口 → 變 INNER）")

## 1.5 其他親戚：RIGHT／FULL／self join

- `RIGHT JOIN`＝方向相反的 LEFT（把表順序換一下就不需要它）。
- `FULL JOIN`＝兩邊都全留、對不上的各自補 NULL（SQLite 3.39+ 原生支援；舊版有經典等價寫法，下一格兩種都示範）——用途：**兩份名單的完整對帳**（誰只在 A、誰只在 B、誰都有）。
- **self join**：同一張表跟自己 join——處理「表內關係」：配對、上下級、前後筆比較。

In [ ]:
# FULL JOIN 對帳：讀書會 A、B 的成員名單（有人兩邊都在、有人只在一邊）
con.executescript("""
DROP TABLE IF EXISTS circle_a; DROP TABLE IF EXISTS circle_b;
CREATE TABLE circle_a(sid TEXT PRIMARY KEY);
CREATE TABLE circle_b(sid TEXT PRIMARY KEY);
INSERT INTO circle_a VALUES ('S001'),('S002'),('S003');
INSERT INTO circle_b VALUES ('S002'),('S003'),('S010');
""")
if sqlite3.sqlite_version_info >= (3, 39):
    print("原生 FULL JOIN（3.39+）：")
    print(q("""SELECT a.sid AS in_a, b.sid AS in_b
               FROM circle_a a FULL JOIN circle_b b ON a.sid = b.sid
               ORDER BY COALESCE(a.sid, b.sid)""").to_string(index=False))
print("\n等價寫法（任何版本；也是面試經典）：LEFT JOIN ∪ 右表獨有")
print(q("""SELECT * FROM (
        SELECT a.sid AS in_a, b.sid AS in_b
        FROM circle_a a LEFT JOIN circle_b b ON a.sid = b.sid
        UNION ALL
        SELECT NULL, b.sid
        FROM circle_b b LEFT JOIN circle_a a ON a.sid = b.sid
        WHERE a.sid IS NULL)
      ORDER BY COALESCE(in_a, in_b)""").to_string(index=False))

In [ ]:
# self join 之一：幫同系學生湊「學伴」配對（a.sid < b.sid 避免重複與自配）
pairs = q("""SELECT a.dept, a.name AS 甲, b.name AS 乙
             FROM student a
             JOIN student b ON a.dept = b.dept AND a.sid < b.sid
             ORDER BY a.dept""")
print(f"可能的同系配對共 {len(pairs)} 組（期望 46：C(8,2)+3×C(4,2) = 28+18）")
pairs.head(6)

In [ ]:
# self join 之二：「同一學期修同一門課」的同學配對——分組名單、共修網絡都是這招
classmates = q("""SELECT a.cid, a.semester, a.sid AS 甲, b.sid AS 乙
                  FROM takes a
                  JOIN takes b ON a.cid = b.cid AND a.semester = b.semester AND a.sid < b.sid
                  ORDER BY a.cid, a.semester, a.sid""")
print(f"共 {len(classmates)} 組同課配對（期望 55）；前 6 組：")
classmates.head(6)

In [ ]:
# 多表實戰：完整授課表（instructor ⋈ teaches ⋈ course）
q("""SELECT i.name AS 教師, c.title AS 課程, te.semester AS 學期
     FROM teaches te
     JOIN instructor i ON te.iid = i.iid
     JOIN course c     ON te.cid = c.cid
     ORDER BY te.semester DESC, i.name LIMIT 8""")

### join 選型決策樹（畫面需求 → join 種類）

```
「每筆 A 配上它的 B」而且沒配到就不要 ────────── INNER JOIN
「每個 A 都要出現（沒有 B 補 0/NULL）」────────── LEFT JOIN（聚合配 COUNT(欄)）
「找從未／沒有 …的 A」──────────────────────── LEFT JOIN + IS NULL（或 NOT EXISTS）
「兩份名單對帳」─────────────────────────────── FULL JOIN（或等價寫法）
「表內配對／前後比較／上下級」────────────────── self join（防重複：a.id < b.id）
```

## 1.6 join 的第二號陷阱：重複計數（報表殺手）

join 會**改變粒度**：student ⋈ takes 之後，一列不再是「一個學生」而是「一筆修課」。
這時 `COUNT(*)`／`SUM(...)` 算的就不是你以為的東西——**專題報表最常見的 bug，沒有之一**。

In [ ]:
# 「統計系有幾位學生修過課？」——join 後 COUNT(*)，人數悄悄變人次
print("COUNT(*)          →", q("""SELECT COUNT(*) c FROM student s
                                   JOIN takes t ON s.sid = t.sid
                                   WHERE s.dept = '統計'""").iloc[0,0],
      f"（期望 25：這是「修課人次」！）")
print("COUNT(DISTINCT sid) →", q("""SELECT COUNT(DISTINCT s.sid) c FROM student s
                                     JOIN takes t ON s.sid = t.sid
                                     WHERE s.dept = '統計'""").iloc[0,0],
      f"（期望 8：這才是人數）")

In [ ]:
# SUM 版更陰險：想「順便帶出老師」，join 條件少對了 semester → 學分數被乘出來
print("S001 的總學分（正確）：",
      q("""SELECT SUM(c.credits) s FROM takes t
           JOIN course c ON t.cid = c.cid WHERE t.sid = 'S001'""").iloc[0,0], f"（期望 12）")
print("多 join 一張 teaches（只對 cid）：",
      q("""SELECT SUM(c.credits) s FROM takes t
           JOIN course c  ON t.cid = c.cid
           JOIN teaches te ON te.cid = c.cid          -- 一門課多學期開課 → 列被複製！
           WHERE t.sid = 'S001'""").iloc[0,0], f"（膨脹成 24！）")
print()
print("防身三招：① join 條件補完整（cid 還要對 semester）")
print("          ② 先在 CTE 把 1:N 的那頭「摺」成一列，再 join")
print("          ③ 數人／數東西一律 COUNT(DISTINCT 主鍵)")
print("心法：join 完先問自己——「現在一列代表什麼？」")

## 1.7 集合運算：UNION／INTERSECT／EXCEPT

把兩個 **SELECT 的結果**當集合疊起來（欄位數與型別要相容）：

| 運算 | 意義 | 去重？ |
|---|---|---|
| `A UNION B` | 聯集 | ✅ 自動去重 |
| `A UNION ALL B` | 串接 | ❌ 保留重複（快，統計次數時用它） |
| `A INTERSECT B` | 交集 | ✅ |
| `A EXCEPT B` | 差集（A 有 B 沒有） | ✅ |

In [ ]:
for label, sql in [
    ("修過 C101 的人數",            "SELECT COUNT(DISTINCT sid) FROM takes WHERE cid='C101'"),
    ("修過 C104 的人數",            "SELECT COUNT(DISTINCT sid) FROM takes WHERE cid='C104'"),
    ("C101 ∪ C104（UNION）",       "SELECT COUNT(*) FROM (SELECT sid FROM takes WHERE cid='C101' UNION SELECT sid FROM takes WHERE cid='C104')"),
    ("UNION ALL（不去重）",         "SELECT COUNT(*) FROM (SELECT sid FROM takes WHERE cid='C101' UNION ALL SELECT sid FROM takes WHERE cid='C104')"),
    ("C101 ∩ C104（INTERSECT）",   "SELECT COUNT(*) FROM (SELECT sid FROM takes WHERE cid='C101' INTERSECT SELECT sid FROM takes WHERE cid='C104')"),
    ("修過 C101 沒修過 C102（EXCEPT）","SELECT COUNT(*) FROM (SELECT sid FROM takes WHERE cid='C101' EXCEPT SELECT sid FROM takes WHERE cid='C102')"),
]:
    print(f"{label:28s} → {con.execute(sql).fetchone()[0]}")
# 期望：13、5、13、18、5、6
# 想想：為什麼 UNION 是 13 不是 13+5=18？（C104 的修課者是 C101 的子集嗎？）

In [ ]:
# UNION ALL 的日常：把「結構相同的兩批資料」疊成一張報表（跨學期、跨分店、跨年度都是它）
q("""SELECT '114-1' AS 學期, COUNT(*) AS 人次, ROUND(AVG(grade),1) AS 平均
     FROM takes WHERE semester = '114-1'
     UNION ALL
     SELECT '114-2', COUNT(*), ROUND(AVG(grade),1)
     FROM takes WHERE semester = '114-2'""")
# 之後學了 GROUP BY semester 一句就能做——但「疊異源資料」（不同表、不同庫）時 UNION ALL 無可取代

### 隨堂練習 A（3 分鐘）

用今天的武器寫出：「**從未**被 I01 老師教過的學生」。先想策略：哪幾張表？anti-join 還是 EXCEPT？

<details><summary>兩種解法</summary>

```sql
-- 解法一：EXCEPT
SELECT sid FROM student
EXCEPT
SELECT DISTINCT t.sid
FROM takes t JOIN teaches te ON t.cid = te.cid AND t.semester = te.semester
WHERE te.iid = 'I01';

-- 解法二：anti-join（LEFT JOIN … IS NULL），或 NOT EXISTS（等等教）
```
被 I01 教過＝「修過 I01 在**該學期**教的課」——join 條件要同時對 cid 與 semester，少一個就是另一個（錯的）題目。剛學的 1.6 陷阱在這裡也埋著：只對 cid 就會把「別學期別人教的同一門課」也算進去。
</details>

### 隨堂練習 A2（3 分鐘，動手）

「每位**有開過課**的老師教過幾門**不同的**課？」——要 join 誰？COUNT 什麼才不會變人次？

<details><summary>參考解</summary>

```sql
SELECT i.name, COUNT(DISTINCT te.cid) AS n_courses
FROM instructor i JOIN teaches te ON i.iid = te.iid
GROUP BY i.iid ORDER BY n_courses DESC;
```
`COUNT(DISTINCT te.cid)`——同一門課開兩學期只算一門（1.6 的教訓現學現用）。
</details>

In [ ]:
# 練習 A2 工作區
# TODO




# 第 2 節（上）：GROUP BY、子查詢與 CTE

## 2.1 GROUP BY 心智模型：分堆 → 每堆濃縮成一列

```
takes 50 列 ──(依 cid 分堆)──►  8 堆 ──(每堆算 COUNT/AVG)──► 8 列
```

三條鐵律：
1. `SELECT` 裡只能放**分組鍵**或**聚合函數**（其他欄位每堆有多個值，放了沒意義——SQLite 不報錯會亂給，要自律！）。
2. `WHERE` 在分組**前**過濾列；`HAVING` 在分組**後**過濾堆。
3. 想「每組的明細列」？GROUP BY 給不了——那是 window functions 的工作（下半場）。

In [ ]:
# 各課程修課人次與平均成績（join 拿課名 + group 濃縮）
q("""SELECT c.cid, c.title, COUNT(*) AS 人次, ROUND(AVG(t.grade), 1) AS 平均
     FROM takes t JOIN course c ON t.cid = c.cid
     GROUP BY c.cid
     ORDER BY 人次 DESC""")

In [ ]:
# WHERE（先砍列）與 HAVING（後砍堆）合體：只看有成績的紀錄，列出「人次 >= 6」的課
q("""SELECT c.title, COUNT(*) AS 人次, ROUND(AVG(t.grade),1) AS 平均
     FROM takes t JOIN course c ON t.cid = c.cid
     WHERE t.grade IS NOT NULL          -- 分組前：只留有成績的列
     GROUP BY c.cid
     HAVING COUNT(*) >= 6               -- 分組後：只留大堆
     ORDER BY 平均 DESC""")
# 沒加 WHERE 條件時人次門檻 6 會選到 3 門課——WHERE/HAVING 順序不同，答案就不同

In [ ]:
# 分組鍵可以是運算式：每「學期」的修課人次與平均
q("""SELECT semester, COUNT(*) AS 人次, ROUND(AVG(grade),1) AS 平均_在修不計
     FROM takes GROUP BY semester ORDER BY semester""")

### 隨堂練習 G（3 分鐘，動手）：GROUP BY ＋ HAVING 一條龍

「各系**有成績**的修課平均，只列出平均 ≥ 75 的系，高分在前」——WHERE 跟 HAVING 各負責哪段？

<details><summary>參考解</summary>

```sql
SELECT s.dept, ROUND(AVG(t.grade), 1) AS avg_grade, COUNT(*) AS n
FROM takes t JOIN student s ON t.sid = s.sid
WHERE t.grade IS NOT NULL          -- 分組前：把 NULL 列先請出去（分母干淨）
GROUP BY s.dept
HAVING AVG(t.grade) >= 75          -- 分組後：砍掉不及門檻的「堆」
ORDER BY avg_grade DESC;
```
WHERE 管「列」、HAVING 管「堆」——順序反了語意就變（把 WHERE 條件寫進 HAVING 雖然常常也對，但分組白做工）。
</details>

In [ ]:
# 練習 G 工作區
# TODO




## 2.2 條件式聚合＝SQL 的樞紐表（pivot）

把「分類」攤成「欄位」：每個欄位一個 `SUM(CASE WHEN …)`。U01 秀場的第 ⑦ 招，現在你看得懂了——
**這招做出來的交叉表，是統計報表的半壁江山**（你的專題「報表 ≥5 張」幾乎必有一張）。

In [ ]:
# 系 × 學期的修課人次交叉表
q("""SELECT s.dept AS 系所,
        SUM(CASE WHEN t.semester = '114-1' THEN 1 ELSE 0 END) AS "114-1",
        SUM(CASE WHEN t.semester = '114-2' THEN 1 ELSE 0 END) AS "114-2",
        SUM(CASE WHEN t.semester = '115-1' THEN 1 ELSE 0 END) AS "115-1",
        COUNT(*) AS 總計
     FROM takes t JOIN student s ON t.sid = s.sid
     WHERE t.semester IN ('114-1','114-2','115-1')
     GROUP BY s.dept ORDER BY 總計 DESC""")

In [ ]:
# 同一張交叉表用 pandas 再算一次（欄位很多、不想手寫 CASE 時就交給 pivot_table）
df_st = q("SELECT s.dept, t.semester FROM takes t JOIN student s ON t.sid = s.sid")
pv = df_st.pivot_table(index="dept", columns="semester", aggfunc="size", fill_value=0)
print(pv.to_string())
print("\n→ SQL 的 CASE 版欄位固定但可進資料庫 view；pandas 版欄位自動長出來。兩邊數字要一致（雙引擎驗證）。")

## 2.3 經典難題：「每系平均成績最高的**那位學生**是誰？」

`GROUP BY dept` 能算出每系的最高平均，但**答不出那是誰**（SELECT 不能放 name）。
這是 SQL 最著名的「每組第一名」問題——傳統解法要 correlated 子查詢繞圈，現代解法用 window functions 三行搞定（下半場揭曉；答案先劇透：企管系 曾柏勳、數學系 許芷瑄、統計系 江美慧、資訊系 吳孟軒）。

## 2.4 子查詢：查詢裡的查詢

| 型 | 長相 | 用途 |
|---|---|---|
| scalar | `WHERE grade > (SELECT AVG(grade) FROM takes)` | 跟「一個算出來的數」比 |
| IN | `WHERE cid IN (SELECT cid FROM course WHERE dept='統計')` | 跟「一組值」比 |
| EXISTS | `WHERE EXISTS (SELECT 1 FROM ... WHERE 關聯到外層)` | 「有沒有」判斷（correlated） |
| FROM 子查詢 | `FROM (SELECT ...) AS x` | 把中間結果當表用（CTE 更好讀） |

In [ ]:
# scalar 子查詢：高於全校平均（81.08）的修課紀錄有幾筆？
q("""SELECT COUNT(*) AS 高於平均筆數
     FROM takes WHERE grade > (SELECT AVG(grade) FROM takes)""")
# 期望：21

In [ ]:
# IN 子查詢：修過「統計系開的課」的外系學生
q("""SELECT DISTINCT s.sid, s.name, s.dept
     FROM student s JOIN takes t ON s.sid = t.sid
     WHERE t.cid IN (SELECT cid FROM course WHERE dept = '統計')
       AND s.dept <> '統計'
     ORDER BY s.dept, s.sid""")

In [ ]:
# EXISTS 正向用法：「修過 C103 的學生」——EXISTS vs IN vs JOIN 三寫法對照（答案一樣、粒度不同）
print("EXISTS：", len(q("""SELECT s.sid FROM student s WHERE EXISTS
                            (SELECT 1 FROM takes t WHERE t.sid = s.sid AND t.cid = 'C103')""")), "人")
print("IN 　　：", len(q("SELECT sid FROM student WHERE sid IN (SELECT sid FROM takes WHERE cid='C103')")), "人")
print("JOIN 　：", len(q("SELECT DISTINCT s.sid FROM student s JOIN takes t ON s.sid=t.sid WHERE t.cid='C103'")), "人")
print("→ 三寫法等價；JOIN 版沒 DISTINCT 就會變人次（1.6 又來了）。「問有沒有」用 EXISTS 語意最乾淨。")

In [ ]:
# NOT EXISTS（correlated：內層引用了外層的 s.sid）：從未修過任何統計系課程的學生
q("""SELECT s.sid, s.name, s.dept
     FROM student s
     WHERE NOT EXISTS (
         SELECT 1 FROM takes t JOIN course c ON t.cid = c.cid
         WHERE t.sid = s.sid AND c.dept = '統計')
     ORDER BY s.dept""")
# 期望：6 位。EXISTS 只問「有沒有」，不在乎內層選什麼——慣例寫 SELECT 1
# 上個單元的伏筆回收：NOT IN 遇到 NULL 會全軍覆沒，NOT EXISTS 天生免疫——找「沒有」優先用它

## 2.5 CTE（`WITH`）：把查詢拆成有名字的步驟

巢狀子查詢一多就變俄羅斯娃娃。CTE 讓你**由上而下說故事**——每一步一個名字，像變數一樣：

```sql
WITH per_student AS (
    SELECT sid, AVG(grade) AS ag FROM takes
    WHERE grade IS NOT NULL GROUP BY sid
)
SELECT s.name, ROUND(a.ag, 2) AS 平均
FROM per_student a JOIN student s ON a.sid = s.sid
ORDER BY a.ag DESC LIMIT 5;
```

之後所有複雜報表（包含你的專題）**一律用 CTE 寫**：好讀、好 debug（每段可以單獨執行）、AI 也比較不會寫錯。

In [ ]:
# CTE 實戰：全校平均成績 Top 5（兩步：先摺每人，再排名）
q("""WITH per_student AS (
        SELECT sid, AVG(grade) AS ag, COUNT(grade) AS n
        FROM takes WHERE grade IS NOT NULL GROUP BY sid)
     SELECT s.sid, s.name, s.dept, ROUND(a.ag,2) AS 平均, a.n AS 科目數
     FROM per_student a JOIN student s ON a.sid = s.sid
     ORDER BY a.ag DESC LIMIT 5""")

In [ ]:
# CTE 的隱藏福利：每一步可以「單獨拉出來跑」——debug 複雜報表的標準流程
step1 = q("SELECT sid, AVG(grade) AS ag, COUNT(grade) AS n FROM takes WHERE grade IS NOT NULL GROUP BY sid")
print("① 先驗中間步驟：per_student 有", len(step1), "列、平均欄有沒有怪值 →",
      f"min={step1.ag.min():.1f}, max={step1.ag.max():.1f}")
print("② 中間對了，再接下一步 join——報表錯了就這樣一層一層剝，三分鐘找到兇手。")
print("③ AI 生的長查詢驗收同理：把它的 CTE 一段一段拆出來跑。")

## 2.6 遞迴 CTE：會「自己長」的查詢

CTE 還有個隱藏技：`WITH RECURSIVE`——**用前一輪的結果再算下一輪**，直到不再產生新列。

```sql
WITH RECURSIVE name(col) AS (
    SELECT 起點                       -- ① anchor：第一輪
    UNION ALL
    SELECT 下一步 FROM name WHERE 繼續條件   -- ② 遞迴：引用自己
)
SELECT * FROM name;
```

兩大用途你的專題都可能遇到：**① 生數列／日曆**（報表補零、產生時段格線）、**② 展開階層**（分類樹、先修鏈、上下級）。

In [ ]:
# 用途①熱身：生成 1..10 的數列（沒有任何表也能查！）
print(q("""WITH RECURSIVE seq(n) AS (
            SELECT 1
            UNION ALL
            SELECT n + 1 FROM seq WHERE n < 10)
          SELECT n, n * n AS n_squared FROM seq""").to_string(index=False))

# 生日曆：2026 年 9 月的每一天＋星期幾（期望 30 列、其中週四 4 天）
cal = q("""WITH RECURSIVE cal(d) AS (
             SELECT '2026-09-01'
             UNION ALL
             SELECT date(d, '+1 day') FROM cal WHERE d < '2026-09-30')
           SELECT d, CASE strftime('%w', d) WHEN '4' THEN '★週四（上課日）' ELSE '' END AS note
           FROM cal""")
print(f"\n生成 {len(cal)} 天；上課日：")
print(cal[cal.note != ''].to_string(index=False))

In [ ]:
# 用途②：展開「先修課程鏈」——修 C102 之前要先修完哪些課？（間接先修也要挖出來）
con.executescript("""
DROP TABLE IF EXISTS prereq;
CREATE TABLE prereq(cid TEXT, needs TEXT);          -- 「cid 需要先修 needs」
INSERT INTO prereq VALUES
  ('C102','C101'), ('C102','C104'),                 -- 迴歸 ← 統計學、機率論
  ('C104','C201'),                                  -- 機率論 ← 微積分
  ('C103','C301');                                  -- 資料庫 ← 程式設計
""")
q("""WITH RECURSIVE need(cid) AS (
        SELECT needs FROM prereq WHERE cid = 'C102'          -- anchor：直接先修
        UNION                                                -- UNION（去重）順便防循環
        SELECT p.needs FROM prereq p JOIN need n ON p.cid = n.cid)   -- 先修的先修…
     SELECT n.cid, c.title FROM need n JOIN course c ON n.cid = c.cid ORDER BY n.cid""")
# 期望 3 門：直接的 C101、C104，加上被 C104 拖出來的 C201——遞迴自動追到底
# 你的專題對照：分類樹、組織階層，都是同一招

# 第 2 節（下）：window functions——統計系的新武器

換大場地：**sales.db**——10 萬筆 2025 年電商訂單（合成資料、固定 seed，人人相同）。先跑 setup：

In [ ]:
#@title 📦 資料準備：合成電商資料 sales.db（100,000 筆訂單，固定種子，人人跑出同一份）
import sqlite3, os
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
N_ORDERS, N_CUST, N_PROD = 100000, 5000, 200

cities  = np.array(["台中","台北","高雄","台南","新竹","桃園"])
cats    = np.array(["飲料","零食","文具","3C配件","生活用品"])

customers = pd.DataFrame({
    "cid":  np.arange(1, N_CUST + 1),
    "cname": [f"顧客{i:04d}" for i in range(1, N_CUST + 1)],
    "city": rng.choice(cities, N_CUST, p=[.3,.25,.15,.12,.1,.08]),
    "age":  rng.integers(18, 70, N_CUST),
})
products = pd.DataFrame({
    "pid": np.arange(1, N_PROD + 1),
    "pname": [f"商品{i:03d}" for i in range(1, N_PROD + 1)],
    "category": rng.choice(cats, N_PROD),
    "price": np.round(rng.lognormal(4.5, 0.6, N_PROD), 0) + 9,
})
# 商品熱門度呈長尾（Zipf）：少數商品貢獻大多數訂單——真實世界就長這樣
pop = rng.zipf(1.3, N_ORDERS * 3)
pop = pop[pop <= N_PROD][:N_ORDERS]
days = rng.integers(0, 365, N_ORDERS)
odate = (np.datetime64("2025-01-01") + days).astype(str)
qty = rng.integers(1, 6, N_ORDERS)
orders = pd.DataFrame({
    "oid": np.arange(1, N_ORDERS + 1),
    "cid": rng.integers(1, N_CUST + 1, N_ORDERS),
    "pid": pop,
    "odate": odate,
    "qty": qty,
})
orders["amount"] = np.round(orders.qty * products.set_index("pid").loc[orders.pid, "price"].to_numpy(), 0)

if os.path.exists("sales.db"):
    os.remove("sales.db")
scon = sqlite3.connect("sales.db")
customers.to_sql("customers", scon, index=False)
products.to_sql("products", scon, index=False)
orders.to_sql("orders", scon, index=False)
scon.execute("CREATE UNIQUE INDEX idx_cust ON customers(cid)")
scon.execute("CREATE UNIQUE INDEX idx_prod ON products(pid)")
scon.commit()

def qs(sql, params=()):
    return pd.read_sql_query(sql, scon, params=params)

print(f"orders {len(orders):,} 列、customers {N_CUST:,} 列、products {N_PROD} 列 → sales.db ✅")

In [ ]:
# 熱身摸資料：規模與長相
print(qs("SELECT COUNT(*) 訂單數, ROUND(SUM(amount)) 總營收, MIN(odate) 起, MAX(odate) 迄 FROM orders").to_string(index=False))
qs("""SELECT o.oid, c.cname, c.city, p.pname, p.category, o.qty, o.amount, o.odate
      FROM orders o JOIN customers c ON o.cid=c.cid JOIN products p ON o.pid=p.pid
      LIMIT 5""")

## 3.1 window function 是什麼？

**聚合但不壓縮列**：每一列保留，旁邊多一欄「站在某個窗口看到的統計量」。

```sql
函數() OVER (PARTITION BY 分堆鍵 ORDER BY 排序鍵 [frame])
--            └ 在哪一堆算        └ 堆內怎麼排   └ 窗框多大（預設到目前列）
```

| 家族 | 函數 | 統計系的話 |
|---|---|---|
| 排名 | `ROW_NUMBER / RANK / DENSE_RANK / NTILE(k)` | 組內名次、分位數分組 |
| 位移 | `LAG / LEAD` | 差分、前後期比較（時間序列！） |
| 分佈 | `PERCENT_RANK / CUME_DIST` | 百分位、經驗分佈函數 |
| 邊界 | `FIRST_VALUE / LAST_VALUE / NTH_VALUE` | 組內第一名／最新一筆 |
| 聚合窗 | `SUM/AVG/COUNT/... OVER (...)` | 累積和、移動平均、佔比 |

pandas 對照：`groupby().transform()`＋`rolling()`＋`rank()` 的合體，但一句 SQL 說完。

In [ ]:
# 排名三兄弟的差別（故意找有同分的：3C配件裡金額最高的訂單們）
qs("""SELECT oid, amount,
        ROW_NUMBER() OVER w AS row_num,     -- 硬編 1,2,3,...（同分也硬分先後）
        RANK()       OVER w AS rnk,         -- 同分同名次，下一名跳號（1,1,3）
        DENSE_RANK() OVER w AS dense_rnk    -- 同分同名次，不跳號（1,1,2）
      FROM orders o JOIN products p ON o.pid = p.pid
      WHERE p.category = '3C配件'
      WINDOW w AS (ORDER BY amount DESC)
      LIMIT 8""")

### 隨堂練習 B（2 分鐘，口頭）：三兄弟選誰？

1. 「班排名」要同分同名次且下一名跳號（88、88、第三名）？
2. 「每組只取一列代表」？
3. 「等第分級：同分必須同級、級數要連續」？

<details><summary>答案</summary>
1. RANK　2. ROW_NUMBER（配 rn=1，同分也硬切出一列）　3. DENSE_RANK。
「取 top-k 明細」用 ROW_NUMBER 最安全——RANK 在同分時會取出超過 k 列。
</details>

In [ ]:
# 「每組 top-k」標準解法：各城市消費總額前 3 名顧客（CTE ＋ ROW_NUMBER ＋ rn<=3）
qs("""WITH cust_total AS (
         SELECT c.city, c.cname, SUM(o.amount) AS total
         FROM orders o JOIN customers c ON o.cid = c.cid
         GROUP BY c.cid),
       ranked AS (
         SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY total DESC) AS rn
         FROM cust_total)
      SELECT city, rn, cname, ROUND(total) AS 總消費
      FROM ranked WHERE rn <= 3
      ORDER BY city, rn""")

In [ ]:
# 回馬槍：解掉 2.3 的難題——每系平均成績最高的學生（univ.db）
q("""WITH avg_by_student AS (
        SELECT s.sid, s.name, s.dept, AVG(t.grade) AS ag
        FROM student s JOIN takes t ON s.sid = t.sid
        WHERE t.grade IS NOT NULL
        GROUP BY s.sid),
      ranked AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY dept ORDER BY ag DESC) AS rn
                 FROM avg_by_student)
     SELECT dept AS 系所, name AS 系冠軍, ROUND(ag, 2) AS 平均
     FROM ranked WHERE rn = 1 ORDER BY dept""")
# GROUP BY 做不到的，window 三行解決。這招你的專題報表一定用得上（各分類冠軍、各時段之最⋯⋯）

## 3.2 LAG／LEAD：跟上一期比（時間序列的差分）

In [ ]:
# 月營收、上月、月成長率——所有儀表板的標配
qs("""WITH monthly AS (
         SELECT strftime('%Y-%m', odate) AS ym, SUM(amount) AS rev
         FROM orders GROUP BY ym)
      SELECT ym AS 月份, ROUND(rev) AS 營收,
             ROUND(LAG(rev) OVER (ORDER BY ym)) AS 上月,
             ROUND((rev - LAG(rev) OVER (ORDER BY ym)) * 100.0
                   / LAG(rev) OVER (ORDER BY ym), 1) AS 成長率pct
      FROM monthly""")
# 第一列的 LAG 是 NULL（沒有上一期）——NULL 的老朋友又出現了，報表要怎麼呈現是你的判斷

In [ ]:
# LEAD 看「下一筆」：同一位顧客這次下單到下次下單隔了幾天（回購間隔——RFM 的 R）
qs("""WITH c9 AS (SELECT cid, odate FROM orders WHERE cid = 9 ORDER BY odate)
      SELECT odate AS 這次,
             LEAD(odate) OVER (ORDER BY odate) AS 下次,
             CAST(julianday(LEAD(odate) OVER (ORDER BY odate)) - julianday(odate) AS INTEGER) AS 間隔天
      FROM c9 LIMIT 8""")
# LAG 往回看、LEAD 往前看；配 PARTITION BY cid 就能一次算全部顧客（lab 挑戰有它）

## 3.3 移動平均：`frame` 子句——窗框開多大

`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` ＝ 本列與前 6 列（7 日窗）。
沒寫 frame 時，有 `ORDER BY` 的預設是「開頭到目前列」＝**累積**。

（細節備查：預設其實是 `RANGE`——**同值的列會整批一起入窗**；移動平均這種「逐列」的窗一律明寫 `ROWS`。）

In [ ]:
# 每日營收 + 7 日移動平均 + 畫圖（日資料太毛躁，移動平均讓趨勢浮出來）
import matplotlib.pyplot as plt
daily = qs("""WITH d AS (SELECT odate, SUM(amount) AS rev FROM orders GROUP BY odate)
              SELECT odate, rev,
                     AVG(rev) OVER (ORDER BY odate ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma7
              FROM d""")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(range(len(daily)), daily.rev, alpha=.35, lw=.8, label="daily revenue")
ax.plot(range(len(daily)), daily.ma7, lw=2, label="7-day moving avg")
ax.set_xlabel("day of 2025"); ax.set_ylabel("revenue"); ax.legend(); ax.set_title("Daily revenue & 7-day MA")
plt.tight_layout(); plt.show()
daily.head(3)

In [ ]:
# FIRST_VALUE 安全、LAST_VALUE 有雷：預設窗框只開到「目前列」！
qs("""WITH d AS (SELECT odate, SUM(amount) AS rev FROM orders GROUP BY odate
                 ORDER BY odate LIMIT 5)
      SELECT odate, ROUND(rev) AS rev,
             ROUND(FIRST_VALUE(rev) OVER (ORDER BY odate)) AS day1,
             ROUND(LAST_VALUE(rev)  OVER (ORDER BY odate)) AS last_naive,
             ROUND(LAST_VALUE(rev)  OVER (ORDER BY odate
                   ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) AS last_correct
      FROM d""")
# last_naive 每列都等於自己——因為預設 frame 到 CURRENT ROW 為止，LAST_VALUE 永遠是自己！
# 要「整組的最後一筆」就把窗開滿（UNBOUNDED ... FOLLOWING）。AI 寫 LAST_VALUE 十次錯八次，就錯在這。

## 3.4 NTILE：把母體切成 k 等分（分位組）

In [ ]:
# 顧客消費力四分位（Q4 = 前 25% 大戶）：每組人數、消費區間、平均——RFM 分析的雛形
qs("""WITH cust AS (SELECT cid, SUM(amount) AS total, COUNT(*) AS n
                     FROM orders GROUP BY cid),
       grouped AS (SELECT *, NTILE(4) OVER (ORDER BY total) AS quartile FROM cust)
      SELECT quartile AS 四分位, COUNT(*) AS 人數,
             ROUND(MIN(total)) AS 下界, ROUND(MAX(total)) AS 上界,
             ROUND(AVG(n), 1) AS 平均下單次數
      FROM grouped GROUP BY quartile""")
# NTILE 不整除時前面的組多 1 人（5000/4 整除所以這裡每組剛好 1250）——報告寫組距時記得檢查

In [ ]:
# PERCENT_RANK 與 CUME_DIST：每位顧客「贏過多少人」——經驗分佈函數的 SQL 寫法
qs("""WITH t AS (SELECT cid, SUM(amount) AS total FROM orders GROUP BY cid)
      SELECT cid, ROUND(total) AS 總消費,
             ROUND(PERCENT_RANK() OVER (ORDER BY total) * 100, 2) AS 贏過pct,
             ROUND(CUME_DIST()    OVER (ORDER BY total) * 100, 2) AS 累積分佈pct
      FROM t ORDER BY total DESC LIMIT 5""")
# 統計對照：PERCENT_RANK = (rank-1)/(n-1)；CUME_DIST = 經驗 CDF 的 F̂(x)。
# 「你的消費贏過 99.98% 的顧客」這種文案，就是這兩個函數寫的

In [ ]:
# 累積和：全年營收過半是哪一天？（帕累托感的統計）
qs("""WITH d AS (SELECT odate, SUM(amount) AS rev FROM orders GROUP BY odate),
       c AS (SELECT odate, SUM(rev) OVER (ORDER BY odate) AS cum,
                    (SELECT SUM(amount) FROM orders) AS total FROM d)
      SELECT MIN(odate) AS 過半日 FROM c WHERE cum >= total * 0.5""")
# 期望：2025-07-02（均勻流量下約在年中；你的專題若有促銷/季節效應，這個日期會很會說故事）

In [ ]:
# 分堆累積＋堆內佔比：各分類的年度累積營收曲線（PARTITION BY 讓每一類自己重新起算）
qs("""WITH m AS (SELECT p.category, strftime('%Y-%m', o.odate) AS ym, SUM(o.amount) AS rev
                 FROM orders o JOIN products p ON o.pid = p.pid
                 GROUP BY p.category, ym)
      SELECT category, ym, ROUND(rev) AS 月營收,
             ROUND(SUM(rev) OVER (PARTITION BY category ORDER BY ym)) AS 累積,
             ROUND(rev * 100.0 / SUM(rev) OVER (PARTITION BY category), 1) AS 佔該類全年pct
      FROM m ORDER BY category, ym LIMIT 14""")
# 同一句裡有兩種窗：有 ORDER BY 的（累積）、沒 ORDER BY 的（整堆總和）——佔比就是「自己 ÷ 整堆」

In [ ]:
# 逐列佔比：這一筆訂單佔「該顧客總消費」幾 %？——GROUP BY 做不到的「每列帶組統計」
qs("""SELECT oid, cid, amount,
        ROUND(SUM(amount) OVER (PARTITION BY cid)) AS 該客總額,
        ROUND(amount * 100.0 / SUM(amount) OVER (PARTITION BY cid), 1) AS 佔比pct
      FROM orders WHERE cid = 9 ORDER BY 佔比pct DESC LIMIT 6""")
# 「單筆金額異常大」的偵測就從這欄開始——佔比 > 50% 的那筆，值得看一眼

In [ ]:
# 遞迴 CTE × LEFT JOIN：報表「補零」——長尾商品 195 的一月銷售，沒賣的日子也要一列 0
qs("""WITH RECURSIVE cal(d) AS (
          SELECT '2025-01-01'
          UNION ALL
          SELECT date(d, '+1 day') FROM cal WHERE d < '2025-01-31')
       SELECT cal.d AS 日期, COALESCE(SUM(o.amount), 0) AS 營收, COUNT(o.oid) AS 筆數
       FROM cal
       LEFT JOIN orders o ON o.odate = cal.d AND o.pid = 195   -- 右表條件放 ON！
       GROUP BY cal.d
       ORDER BY cal.d LIMIT 10""")
# 一月 31 天中這個商品只有 0 天有生意、31 天掛零——直接 GROUP BY orders 那些 0 會消失，
# 圖表的 X 軸就會缺日子。日曆表（遞迴生成）× LEFT JOIN 是報表基本功；你的專題畫時間序列圖請照抄這招。

In [ ]:
# 週節奏：每週營收趨勢（strftime('%W') 當分組鍵）——月太粗、日太吵時的中庸解析度
qs("""SELECT strftime('%W', odate) AS week_no, COUNT(*) AS 訂單數, ROUND(SUM(amount)) AS 營收
      FROM orders GROUP BY week_no ORDER BY week_no LIMIT 8""")
# 你的專題挑報表解析度：日（吵）／週（剛好）／月（穩）——畫出來比一比就知道該用哪個

## 3.5 GROUP BY vs window：什麼時候用誰？

| 你要的東西 | 用 |
|---|---|
| 每組**一列**摘要（各系人數、月營收） | `GROUP BY` |
| **每列**帶上組統計（這筆訂單佔該客總額幾 %） | window |
| 每組**前 k 名明細** | window（`ROW_NUMBER` + `rn<=k`） |
| 期別差分、移動平均、累積 | window（`LAG`／frame） |
| 百分位、經驗分佈 | window（`PERCENT_RANK`／`CUME_DIST`） |

兩者常常合體：先 `GROUP BY` 摺一層，再開 window（本節每個例子都是這個 pattern）。

## 3.6 VIEW：把常用查詢封裝成「虛擬表」

```sql
CREATE VIEW v_monthly AS
SELECT strftime('%Y-%m', odate) AS ym, COUNT(*) AS orders, SUM(amount) AS rev
FROM orders GROUP BY ym;
```
- view **不存資料**，查它時即時展開——永遠是最新結果。
- 用途：報表定型（你的專題 5 張報表可以各包成 view）、簡化下游查詢、限縮欄位當權限層。

In [ ]:
scon.execute("DROP VIEW IF EXISTS v_monthly")
scon.execute("""CREATE VIEW v_monthly AS
    SELECT strftime('%Y-%m', odate) AS ym, COUNT(*) AS n_orders, SUM(amount) AS rev
    FROM orders GROUP BY ym""")
print(qs("SELECT * FROM v_monthly ORDER BY ym LIMIT 4").to_string(index=False))
print()
print(qs("SELECT name, type FROM sqlite_master WHERE type IN ('table','view')").to_string(index=False))

## 3.7 pandas ↔ SQL：對照表與雙向搬運

| pandas | SQL |
|---|---|
| `df[df.x > 3]` | `WHERE x > 3` |
| `df[["a","b"]]`／`assign` | `SELECT a, b, 運算式` |
| `merge(a, b, on=..., how="left")` | `LEFT JOIN ... ON` |
| `groupby().agg()` | `GROUP BY` |
| `groupby().transform()`／`rolling()`／`rank()` | window functions |
| `pivot_table` | 條件式聚合（`SUM(CASE …)`） |
| `sort_values().head(k)` | `ORDER BY ... LIMIT k` |
| `drop_duplicates()` | `DISTINCT` |
| `pd.concat` | `UNION ALL` |

**分工哲學**：資料的家在資料庫（永續、約束、多人）；SQL 負責「搬對的資料出來」（過濾、join、聚合到剛好的粒度）；pandas 負責統計精算與畫圖。

In [ ]:
# SQL 撈 → pandas 算：月營收的敘述統計與變異係數（SQL 沒有現成 std，pandas 一行）
df_m = pd.read_sql_query("SELECT * FROM v_monthly", scon)
print(df_m.rev.describe().round(0).to_string())
print("變異係數 CV =", round(df_m.rev.std() / df_m.rev.mean(), 4))

# pandas 算完 → 寫回資料庫（分析結果也值得有個家）
df_m.assign(rev_share=(df_m.rev / df_m.rev.sum()).round(4)) \
    .to_sql("monthly_summary", scon, if_exists="replace", index=False)
print("\n寫回 monthly_summary ✅ →", scon.execute("SELECT COUNT(*) FROM monthly_summary").fetchone()[0], "列")

## 3.8 DuckDB 十分鐘：分析型引擎初見面

| | SQLite | DuckDB |
|---|---|---|
| 定位 | 交易型（OLTP）：一次讀寫幾列 | 分析型（OLAP）：整欄掃描聚合 |
| 儲存 | 列式（row store） | **欄式（column store）＋向量化執行** |
| 絕活 | 約束、交易、單檔應用 | 大表聚合快 10–100 倍；**直接查 CSV／Parquet／DataFrame** |

安裝一次 `pip install duckdb`；語法幾乎同源。U08 兩引擎正面對決並解釋原理；今天先看它最迷人的三招。

In [ ]:
# 第一招：直接對 pandas DataFrame 下 SQL（orders / products 是 setup 留下的 DataFrame）
import sys, importlib.util, subprocess
if importlib.util.find_spec("duckdb") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
import duckdb

print(duckdb.sql("""
    SELECT p.category, ROUND(SUM(o.amount)) AS rev, COUNT(*) AS n
    FROM orders o JOIN products p ON o.pid = p.pid    -- 這裡的 orders/products 是「記憶體裡的 DataFrame」！
    GROUP BY p.category ORDER BY rev DESC
""").df().to_string(index=False))

In [ ]:
# 第二招：CSV 直接當表查——連整個資料夾的散檔都能用萬用字元一次查
orders[orders.odate <  "2025-07-01"].to_csv("sales_h1.csv", index=False)
orders[orders.odate >= "2025-07-01"].to_csv("sales_h2.csv", index=False)

print(duckdb.sql("SELECT COUNT(*) AS h1 FROM read_csv_auto('sales_h1.csv')").df().to_string(index=False))
print(duckdb.sql("""SELECT strftime(odate, '%Y-%m') AS ym, COUNT(*) AS n
                    FROM read_csv_auto('sales_h*.csv')       -- 兩個檔案一起查！
                    GROUP BY ym ORDER BY ym DESC LIMIT 3""").df().to_string(index=False))
# 「每月一個 CSV 的資料夾」是統計工作的日常——不用先合併，glob 直接查

In [ ]:
# 第三招：Parquet——分析界的標準列式檔案格式（壓縮小、讀取快），DuckDB 原生讀寫
duckdb.sql("COPY (SELECT * FROM orders) TO 'orders.parquet' (FORMAT PARQUET)")
import os
print(f"CSV 版約 {os.path.getsize('sales_h1.csv')/1e6 + os.path.getsize('sales_h2.csv')/1e6:.1f} MB "
      f"→ Parquet {os.path.getsize('orders.parquet')/1e6:.1f} MB（列式＋壓縮）")
print(duckdb.sql("""SELECT COUNT(*) AS n, ROUND(SUM(amount)) AS rev
                    FROM read_parquet('orders.parquet')""").df().to_string(index=False))
# 下載資料集看到 .parquet 別慌——它比 CSV 更快更小，duckdb.sql 一行就讀

## 3.9【AI 協作】把「中文分析需求」變 SQL——含驗收

**好 prompt 的模板**（把 schema 一起貼給 AI，成功率差三倍）：

> 我的 SQLite 資料庫 schema 如下：
> `orders(oid, cid→customers, pid→products, odate, qty, amount)`、`customers(cid, cname, city, age)`、`products(pid, pname, category, price)`
> 請寫一句 SQL（用 CTE）：各年齡層（18–29／30–44／45+）在各商品分類的消費總額，並算出各年齡層內的分類佔比，佔比用 window function。

**驗收 SOP**：
1. 跑得過嗎？欄位名是不是它幻想出來的？
2. **抽樣對照**：用 pandas 獨立算一次同一個數字，兩邊要相等（下一格示範這個黃金驗證法）
3. **粒度盤問**（1.6 的陷阱）：「這句 join 之後一列代表什麼？COUNT 會不會數成人次？」
4. 追問：「為什麼佔比要用 window 不是再 GROUP BY 一次？」——它講得清楚，你才算學到

In [ ]:
# 黃金驗證法：同一題 SQL vs pandas 各算一次，數字對得上才放行
sql_ans = qs("""WITH t AS (SELECT c.city, SUM(o.amount) AS rev
                          FROM orders o JOIN customers c ON o.cid = c.cid GROUP BY c.city)
               SELECT city, rev, ROUND(rev * 100.0 / SUM(rev) OVER (), 2) AS pct
               FROM t ORDER BY rev DESC""")

df_all = pd.read_sql_query("""SELECT c.city, o.amount FROM orders o
                              JOIN customers c ON o.cid = c.cid""", scon)
pd_ans = (df_all.groupby("city").amount.sum().sort_values(ascending=False))
pd_pct = (pd_ans * 100 / pd_ans.sum()).round(2)

match_ok = all(abs(sql_ans.set_index("city").pct - pd_pct) < 0.01)
print(sql_ans.to_string(index=False))
print("\nSQL 與 pandas 交叉驗證：", "✅ 一致" if match_ok else "❌ 不一致，快抓 bug")

# 實作時間（25 分鐘）：sales.db 分析 8 題

規則同上個單元：先自己寫，再開解答核對。1–4 基本、5–8 進階（window）。

In [ ]:
# 第 1 題：列出金額最高的 5 筆訂單：顧客名、商品名、金額、日期（三表 join）
# 期望：5 列，由高到低




In [ ]:
# 第 2 題：各城市的總營收，由高到低（join customers）
# 期望：6 列，台中最高




In [ ]:
# 第 3 題：下單「次數」達 30 次以上的顧客有幾位？（GROUP BY + HAVING 之後再數）
# 期望：124




In [ ]:
# 第 4 題：每月訂單數與營收（一句 GROUP BY；分組鍵用 strftime）
# 期望：12 列




In [ ]:
# 第 5 題：各商品分類「營收最高的那一個商品」（CTE + ROW_NUMBER，每組 top-1）
# 期望：5 列（每分類一列）




In [ ]:
# 第 6 題：各月營收與「上月差額」（LAG）
# 期望：12 列，1 月的差額是 NULL




In [ ]:
# 第 7 題：把「顧客總消費」分成十分位（NTILE(10)），列出第 10 組（前 10% 大戶）的人數與消費下界
# 期望：1 列，人數 500




In [ ]:
# 第 8 題：12 月「完全沒有下單」的顧客有幾位？（LEFT JOIN，條件放 ON——今天的陷阱考點）
# 期望：962




<details><summary>📖 參考解答（1–8）</summary>

```sql
-- 1
SELECT c.cname, p.pname, o.amount, o.odate
FROM orders o JOIN customers c ON o.cid=c.cid JOIN products p ON o.pid=p.pid
ORDER BY o.amount DESC LIMIT 5;
-- 2
SELECT c.city, ROUND(SUM(o.amount)) AS rev
FROM orders o JOIN customers c ON o.cid=c.cid
GROUP BY c.city ORDER BY rev DESC;
-- 3
SELECT COUNT(*) FROM (
  SELECT cid FROM orders GROUP BY cid HAVING COUNT(*) >= 30);
-- 4
SELECT strftime('%Y-%m', odate) AS ym, COUNT(*) AS n, ROUND(SUM(amount)) AS rev
FROM orders GROUP BY ym ORDER BY ym;
-- 5
WITH s AS (SELECT p.category, p.pname, SUM(o.amount) AS rev
           FROM orders o JOIN products p ON o.pid=p.pid GROUP BY p.pid),
     r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY rev DESC) rn FROM s)
SELECT category, pname, ROUND(rev) FROM r WHERE rn = 1 ORDER BY category;
-- 6
WITH m AS (SELECT strftime('%Y-%m', odate) ym, SUM(amount) rev FROM orders GROUP BY ym)
SELECT ym, ROUND(rev), ROUND(rev - LAG(rev) OVER (ORDER BY ym)) AS diff FROM m;
-- 7
WITH t AS (SELECT cid, SUM(amount) total FROM orders GROUP BY cid),
     d AS (SELECT *, NTILE(10) OVER (ORDER BY total) g FROM t)
SELECT g, COUNT(*) AS 人數, ROUND(MIN(total)) AS 下界 FROM d WHERE g = 10 GROUP BY g;
-- 8
SELECT COUNT(*) FROM customers c
LEFT JOIN orders o ON c.cid = o.cid AND strftime('%m', o.odate) = '12'
WHERE o.oid IS NULL;
```
</details>

In [ ]:
# 挑戰 9（選做）：每月「新客」數——該月第一次下單的顧客有幾位？
# 提示：先算每位顧客的首購日（GROUP BY + MIN），再按月數人
# 期望：只有 6 個月有新客、加總 = 5000——平均每人 20 單，幾個月內就人人買過了，
#       之後的月份根本「沒有新客那一列」。想讓掛零的月份也現身？用今天的日曆補零技巧！




In [ ]:
# 挑戰 10（選做）：哪些月份「連續第二個月成長」？（LAG 用兩次：跟上月比、上月跟上上月比）
# 期望：2 個月




<details><summary>📖 挑戰題解答</summary>

```sql
-- 9
WITH first_buy AS (SELECT cid, MIN(odate) AS d0 FROM orders GROUP BY cid)
SELECT strftime('%Y-%m', d0) AS ym, COUNT(*) AS 新客數
FROM first_buy GROUP BY ym ORDER BY ym;

-- 10
WITH m AS (SELECT strftime('%Y-%m', odate) ym, SUM(amount) rev FROM orders GROUP BY ym),
     g AS (SELECT ym, rev,
                  LAG(rev)    OVER (ORDER BY ym) AS p1,
                  LAG(rev, 2) OVER (ORDER BY ym) AS p2
           FROM m)
SELECT ym FROM g WHERE rev > p1 AND p1 > p2;
```
挑戰 9 是 cohort 分析的第一步（每月新客 → 之後追他們的留存）——注意沒新客的月份整列消失，補零技巧再度登場；挑戰 10 的「連續 K 期」思路在偵測趨勢、連續達標時很常用。
</details>

# ★ 專題指派公布（課末 10 分鐘）

- **指派名單今日公布於課程平台**；**你的題目完整規格**（情境、資料表建議、必備功能、進階選項、報表點子、demo 亮點）同步發給。
- 每一題都是「真的能操作的資料庫應用」——情境各異，但**共同要求 10 條人人一樣**（那才是評分主體）。
- **統一技術棧**：Colab ＋ SQLite ＋ **Gradio**（U05 開教）＋ pandas/matplotlib。
- **被指派同一題的同學必須差異化**：資料各自合成（不同 seed 與分佈假設）＋ 進階功能各選不同項＋版面報表自由發揮。
- **今晚回家**：精讀 [`projects.md`](https://github.com/chang-ye-tu/db/blob/master/projects.md) 的「共同要求 10 條」＋你自己的題目規格；下個單元設計課直接拿你的題目開工（工作坊：名詞動詞分析 → ER 圖 → DDL 草稿）。
- 教師示範專題（圖書館借閱管理、個人記帳分析——**非指派題目**）：U05 播放影片＋公開完整程式，當作完成度標竿。

## 自主練習（非繳交）＆ 專題進度建議

**sales.db 進階自主練習**（10 題；setup 格照抄即可重建同一份資料。練完這輪，專題的報表你就寫得動了）：

1. 各分類的訂單數、總營收、**佔全站營收比例**（window）
2. 各城市顧客的「平均客單價」（總營收/訂單數），由高到低
3. 只列出「平均客單價高於全站平均」的城市（子查詢）
4. 每週（`strftime('%W')`）營收趨勢
5. 各分類**營收前 2 名**商品（每組 top-k）
6. 每月營收的**3 個月移動平均**（frame）
7. 各顧客年齡層（18–29／30–44／45+，用 CASE）× 分類的消費交叉表（條件式聚合）
8. 消費金額**前 1%** 的大戶名單與人數（NTILE(100) 或子查詢皆可）
9. 每位顧客「相鄰兩次下單的平均間隔天數」全站平均（LAG ＋ julianday）
10. 自由發揮：問一個你自己好奇的問題，寫 SQL 回答並解讀

規範建議：一律用 CTE 組織；每題先一行說明「這題在問什麼」。**不用繳交**——但這十題就是專題報表的健身房。

**專題進度建議（本單元）**：精讀自己的題目與共同要求 10 條；把 lab 8 題＋挑戰題補完；拿一張紙為你的題目列「名詞清單」（會員？活動？訂單？），下個單元工作坊直接開工。

# 本單元你應該帶走

1. join 是正規化的另一半：INNER 取交集、LEFT 保左全、FULL 對帳、self 處理表內關係；**右表過濾放 ON，結果過濾放 WHERE**。
2. **join 會改變粒度**：接完先問「一列代表什麼」——數人用 `COUNT(DISTINCT)`，或先摺再接。
3. `GROUP BY` 分堆濃縮，`HAVING` 砍堆；`SUM(CASE …)` 條件式聚合＝交叉表。
4. 子查詢四型與 CTE——複雜報表一律 `WITH` 分步說故事（每步可單獨 debug）；**遞迴 CTE** 生日曆補零、展開階層。
5. window functions：`OVER (PARTITION BY … ORDER BY … frame)`——排名、top-k、差分、移動平均、分位、累積、佔比；`LAST_VALUE` 記得開滿窗。
6. SQL 撈、pandas 算、圖表畫——加上 DuckDB 直查 DataFrame／CSV／Parquet 的絕活。
7. **AI 產出要「雙引擎交叉驗證」＋「粒度盤問」**：數字對上、粒度講得清才放行。
8. 你的專題題目已定——去讀規格書！

**下個單元**：資料庫設計——ER 圖、正規化，然後**現場為你自己的題目設計 schema**（工作坊）。讀物：Silberschatz ch6–7；Ullman ch3–4。

---
## 附錄 A：本單元 cheatsheet

```sql
-- join
FROM a JOIN b ON a.k = b.k                  -- inner
FROM a LEFT JOIN b ON a.k = b.k AND b.型別過濾   -- 右表條件放 ON！
WHERE b.k IS NULL                           -- anti-join：找「沒有」
FROM t1 a JOIN t1 b ON ... AND a.id < b.id  -- self join 防重複配對
FROM a FULL JOIN b ON a.k = b.k             -- 對帳（3.39+；等價寫法見 1.5）
-- join 後數人：COUNT(DISTINCT 主鍵)！

-- 分組
GROUP BY 鍵 HAVING 聚合條件
SUM(CASE WHEN 條件 THEN 1 ELSE 0 END)       -- 條件式聚合（樞紐）

-- 集合
UNION / UNION ALL / INTERSECT / EXCEPT

-- CTE
WITH step1 AS (...), step2 AS (SELECT ... FROM step1) SELECT ... FROM step2;
WITH RECURSIVE cal(d) AS (SELECT '起日' UNION ALL
     SELECT date(d,'+1 day') FROM cal WHERE d < '迄日')      -- 日曆／數列／階層

-- window
ROW_NUMBER() OVER (PARTITION BY g ORDER BY x DESC)          -- 組內名次
LAG(x) OVER (ORDER BY t)  /  LEAD(x) OVER (ORDER BY t)      -- 上一期／下一期
AVG(x) OVER (ORDER BY t ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  -- 移動平均
SUM(x) OVER (ORDER BY t)                                    -- 累積
SUM(x) OVER (PARTITION BY g ORDER BY t)                     -- 分堆累積
NTILE(4) OVER (ORDER BY x)                                  -- 四分位組
PERCENT_RANK() OVER (ORDER BY x)                            -- 百分位（0~1）
x * 100.0 / SUM(x) OVER ()                                  -- 佔比
x * 100.0 / SUM(x) OVER (PARTITION BY g)                    -- 堆內佔比
LAST_VALUE(x) OVER (ORDER BY t
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)  -- 開滿窗才是真的最後
```

## 附錄 B：圖表要顯示中文？（專題用得上）

matplotlib 在 Colab 預設沒有中文字型，中文會變 □□。你的專題圖表想用中文標籤時，抄 U01「刀四」那格的字型 bootstrap（apt 安裝 Noto CJK＋addfont）到 notebook 開頭跑一次即可。

## 附錄 C：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| §1 join 與集合運算 | §4.1–4.2 | §6.2–6.3 |
| §2 聚合、子查詢、CTE | §3.7–3.9、§5.1（遞迴另見 §5.4） | §6.3–6.5 |
| §3 window functions | §5.5 | ——（課本較舊，以講義為準） |
| view | §4.2 | §8.1 |